In [153]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn
import sklearn


In [154]:
df = pd.read_csv("test.csv")
df.head()
X, y = df["X"], df["y"]

X = torch.from_numpy(X.to_numpy()).type(dtype=torch.float32)
y = torch.from_numpy(y.to_numpy()).type(dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [155]:
class Regularization_Term(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(1))
        self.bias = nn.Parameter(torch.rand(1))
    def forward(self,X):
        return self.weight*X+self.bias

In [156]:
def Acc_Func(y_true, y_pred):
    return (torch.eq(y_true, y_pred).sum().item() / len(y_true)) * 100

In [157]:
model30 = Regularization_Term()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model30.parameters(), lr=0.0001)
epochs = 2000

lambda_L2 = 0.01
lambda_L1 = 0.01
for epoch in range(epochs):
    model30.train()
    y_preds = model30(X_train)
    loss = (
        loss_fn(y_preds, y_train)
        + lambda_L1 * torch.sum(torch.abs(model30.weight))
        + lambda_L2 * torch.sum(model30.weight**2)
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model30.eval()
    with torch.inference_mode():
        test_preds = model30(X_test)
        test_loss = loss_fn(test_preds, y_test)

        # Round predictions to nearest integer
        y_pred_round = torch.round(test_preds)

        acc = Acc_Func(y_test, y_pred_round)
        if epoch % 20 == 0:
            print(
                f"Epoch: {epoch}, Loss: {loss:.6f}, Test Loss: {test_loss:.6f}, Accuracy: {acc:.2f}%"
            )

Epoch: 0, Loss: 5184.728027, Test Loss: 377.726624, Accuracy: 0.00%
Epoch: 20, Loss: 0.257258, Test Loss: 0.234072, Accuracy: 70.00%
Epoch: 40, Loss: 0.256882, Test Loss: 0.233627, Accuracy: 70.00%
Epoch: 60, Loss: 0.256509, Test Loss: 0.233186, Accuracy: 70.00%
Epoch: 80, Loss: 0.256136, Test Loss: 0.232740, Accuracy: 70.00%
Epoch: 100, Loss: 0.255765, Test Loss: 0.232298, Accuracy: 70.00%
Epoch: 120, Loss: 0.255392, Test Loss: 0.231856, Accuracy: 70.00%
Epoch: 140, Loss: 0.255023, Test Loss: 0.231419, Accuracy: 70.00%
Epoch: 160, Loss: 0.254653, Test Loss: 0.230977, Accuracy: 70.00%
Epoch: 180, Loss: 0.254285, Test Loss: 0.230538, Accuracy: 70.00%
Epoch: 200, Loss: 0.253916, Test Loss: 0.230102, Accuracy: 70.00%
Epoch: 220, Loss: 0.253548, Test Loss: 0.229663, Accuracy: 70.00%
Epoch: 240, Loss: 0.253180, Test Loss: 0.229227, Accuracy: 70.00%
Epoch: 260, Loss: 0.252814, Test Loss: 0.228793, Accuracy: 70.00%
Epoch: 280, Loss: 0.252448, Test Loss: 0.228358, Accuracy: 70.00%
Epoch: 300, 

In [158]:
print(model30.weight.item())
print(model30.bias.item())

2.0122010707855225
0.17166392505168915


In [159]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test.numpy(),test_preds.numpy())
print(f"R2 Score: {r2:.4f}")

R2 Score: 0.9999


In [160]:
for name, param in model30.named_parameters():
    if param.grad is not None:
        print(name, param.grad)

weight tensor([0.0062])
bias tensor([-0.3939])


In [161]:
from sklearn.linear_model import Ridge

ridge_reg = Ridge(alpha=1, solver="cholesky")
ridge_reg.fit(X.reshape(-1, 1), y)

Ridge(alpha=1, solver='cholesky')

In [162]:
print(ridge_reg.predict([[2]]))
print(model30.weight.item()*2+model30.bias.item())

[5.0011641]
4.196066066622734


In [163]:
from sklearn.linear_model import  ElasticNet
elastic_net = ElasticNet(alpha=1,l1_ratio=0.5)
elastic_net.fit(X.reshape(-1,1),y)



ElasticNet(alpha=1)

In [164]:
elastic_net.predict(X=[[2]])

array([5.08725637])

In [165]:
best_loss = float("inf")
patience = 5
counter = 0

In [166]:
for epoch in range(epochs):
    model30.train()
    y_preds = model30(X_train)

    loss = (
        loss_fn(y_preds, y_train)
        + lambda_L1 * torch.sum(torch.abs(model30.weight))
        + lambda_L2 * torch.sum(model30.weight**2)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model30.eval()
    with torch.inference_mode():
        test_preds = model30(X_test)
        test_loss = loss_fn(test_preds, y_test)

    if test_loss < best_loss:
        best_loss = test_loss
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Train Loss {loss:.4f} | Test Loss {test_loss:.4f}")

Epoch 0 | Train Loss 0.2235 | Test Loss 0.1939
Epoch 20 | Train Loss 0.2232 | Test Loss 0.1935


Epoch 40 | Train Loss 0.2228 | Test Loss 0.1932
Epoch 60 | Train Loss 0.2225 | Test Loss 0.1928
Epoch 80 | Train Loss 0.2222 | Test Loss 0.1924
Epoch 100 | Train Loss 0.2219 | Test Loss 0.1921
Epoch 120 | Train Loss 0.2216 | Test Loss 0.1917
Epoch 140 | Train Loss 0.2213 | Test Loss 0.1913
Epoch 160 | Train Loss 0.2210 | Test Loss 0.1910
Epoch 180 | Train Loss 0.2207 | Test Loss 0.1906
Epoch 200 | Train Loss 0.2204 | Test Loss 0.1903
Epoch 220 | Train Loss 0.2201 | Test Loss 0.1899
Epoch 240 | Train Loss 0.2198 | Test Loss 0.1895
Epoch 260 | Train Loss 0.2195 | Test Loss 0.1892
Epoch 280 | Train Loss 0.2192 | Test Loss 0.1888
Epoch 300 | Train Loss 0.2189 | Test Loss 0.1885
Epoch 320 | Train Loss 0.2186 | Test Loss 0.1881
Epoch 340 | Train Loss 0.2183 | Test Loss 0.1877
Epoch 360 | Train Loss 0.2180 | Test Loss 0.1874
Epoch 380 | Train Loss 0.2177 | Test Loss 0.1870
Epoch 400 | Train Loss 0.2174 | Test Loss 0.1867
Epoch 420 | Train Loss 0.2171 | Test Loss 0.1863
Epoch 440 | Train Loss 